In [ ]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d
)
from utils.inference import predict_tiled_unet

from utils.feature_extraction import (
    calculate_distance_to_root_surface,
    extract_nuclei_features_per_marker,
    extract_nuclei_depth
)

from utils.fret import compute_fret_ratios

import tifffile
import napari
import torch



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2026-04-21 15:19:24,385 [INFO] WRITING LOG OUTPUT TO C:\Users\adiez_cmic\.cellpose\run.log
2026-04-21 15:19:24,385 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0
2026-04-21 15:19:24,505 [INFO] ** TORCH CUDA version installed and working. **
2026-04-21 15:19:24,507 [INFO] ** TORCH CUDA version installed and working. **
2026-04-21 15:19:24,507 [INFO] >>>> using GPU (CUDA)
2026-04-21 15:19:25,504 [INFO] >>>> loading model C:\Users\adiez_cmic\.cellpose\models\cpsam


In [2]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor, DD)
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor, DA – FRET signal)
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor, AA) – used for nuclei segmentation (and optionally for correction)
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 1 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 0

#TODO: Emission ratio (standard FRET readout, matching ImageJ plugin logic):
# FRET ratio = (donor-excited acceptor emission) / (donor-excited donor emission)
#            = edCitrine_FRET / edCerulean_CTRL
#
# Note:
# - Ratio is computed per object using summed intensities (ΣDA / ΣDD)
# - Only valid where both donor and acceptor intensities > 0
# - Saturated pixels should be excluded and the same mask (nuclei_labels) applied to both channels
# - No bleed-through or direct excitation correction is applied in this raw ratio


In [3]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [4]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (121, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (149, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (126, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (120, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (174, 4, 256, 1024)
Image name: Col0 8, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (111, 4, 256, 1024)
Image name: Col0 9, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (130, 4, 256, 1024)


In [5]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

2026-04-21 15:19:28,763 [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


[<Image layer 'edCerulean_CTRL' at 0x25d042d2dd0>,
 <Image layer 'edCitrine_FRET' at 0x25d0405a200>,
 <Image layer 'edCitrine_CTRL' at 0x25c65cedc00>,
 <Image layer 'brightfield' at 0x25c65d2b760>]

In [6]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x25c6619a980>

In [7]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_sorb 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 1 ...loading


In [8]:
# Ensure output directory for this container's 3D root mask
root_mask_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="root_mask")
print(f"3D Root Mask directory: {root_mask_dir}")

# Load precomputed root mask when available; otherwise generate and store it
smooth_root_3d = load_precomputed_results_if_available(root_mask_dir, lif_image_name, results_type="root_mask")

if smooth_root_3d is not None:
    print(f"3D root mask already calculated for: {lif_image_name} ...loading")
    # Add the precomputed mask to the napari viewer
    viewer.add_image(smooth_root_3d, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

else:
    # Predict root cell boundary probability maps using a pre-trained UNet3D model
    root_pmaps = predict_tiled_unet(
        raw=sim_fluo_cell_walls,
        input_layout="ZYX",
        model_dir=MODEL_DIR,
        patch=(80, 160, 160),
        patch_halo=(8, 16, 16),
        stride_ratio=0.75,
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_amp=True,
    )

    # root_pmaps: (C_out, Z, Y, X)
    viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

    # Generate a rough 3D root mask
    filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)
    # Fill internal gaps inside rough 3D root mask
    filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)
    # Smooth root outer surface to remove small protrusions
    smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)
    # Create path for root mask (used only when saving a newly computed prediction)
    root_mask_path = root_mask_dir / f"{lif_image_name}_root_mask.tif"
    # Save the processed mask
    tifffile.imwrite(root_mask_path, smooth_root_3d)

3D Root Mask directory: ..\raw_data\root_mask\20260129_ABACUS timepoints_sorb 3h
3D root mask already calculated for: Col0 1 ...loading


In [9]:
# Ensure output directory for this container's nuclei depth_map
depth_map_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="depth_map")
print(f"Nuclei depth map directory: {depth_map_dir}")

# Load precomputed depth map when available; otherwise generate and store it
nuclei_depth_map = load_precomputed_results_if_available(depth_map_dir, lif_image_name, results_type="depth_map")
is_flooded = False
flooded_planes = []

if nuclei_depth_map is not None:
    print(f"Nuclei depth map already calculated for: {lif_image_name} ...loading")
    # Add the precomputed map to Napari
    viewer.add_image(nuclei_depth_map, name="nuclei_depth_normalized", colormap="viridis", blending="additive")

else:
    # Calculate distance from each nuclei centroid to the root surface
    # This will be used to approximate to which tissue layer each nucleus belongs
    nuclei_depth_map, is_flooded, flooded_planes = calculate_distance_to_root_surface(
        nuclei_labels,
        smooth_root_3d,
        pad_full_root=False,
        visualize=True,
    )
    # Create path for nuclei depth map (used only when saving a newly computed prediction)
    nuclei_depth_path = depth_map_dir / f"{lif_image_name}_depth_map.tif"
    # Save the calculated depth map
    tifffile.imwrite(nuclei_depth_path, nuclei_depth_map)

print(f"Flood fill applied: {is_flooded}; flooded planes: {flooded_planes}")

Nuclei depth map directory: ..\raw_data\depth_map\20260129_ABACUS timepoints_sorb 3h
Nuclei depth map already calculated for: Col0 1 ...loading
Flood fill applied: False; flooded planes: []


In [ ]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            "flood_filled": is_flooded,
            }

# Extract morphological, depth and intensity features per marker
props_df = extract_nuclei_features_per_marker(nuclei_labels, lif_image, MARKERS, descriptor_dict)
depth_df = extract_nuclei_depth(nuclei_labels, nuclei_depth_map)
props_df = props_df.merge(depth_df, on="label")

# Calculate FRET ratios
props_df = compute_fret_ratios(props_df)
props_df

,lif_container_id,lif_image_name,flood_filled,label,area,area_bbox,area_convex,area_filled,axis_major_length,axis_minor_length,...,edCitrine_FRET_min_int,edCitrine_FRET_max_int,edCitrine_FRET_std_int,edCitrine_FRET_sum_int,edCitrine_CTRL_mean_int,edCitrine_CTRL_min_int,edCitrine_CTRL_max_int,edCitrine_CTRL_std_int,edCitrine_CTRL_sum_int,depth
0,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,1,344.0,648.0,405.0,344.0,18.542413,5.664366,...,0.0,107.0,26.633593,13110.0,220.125000,31.0,458.0,120.852154,75723.0,0.060050
1,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2,466.0,1024.0,524.0,466.0,16.681678,6.095668,...,0.0,64.0,14.610232,10344.0,142.899142,21.0,328.0,80.896244,66591.0,0.047228
2,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,3,440.0,800.0,496.0,440.0,16.776578,5.059086,...,2.0,69.0,16.652364,11941.0,170.643182,29.0,366.0,90.749918,75083.0,0.047228
3,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,4,311.0,686.0,362.0,311.0,14.512868,6.100066,...,0.0,103.0,21.046501,11677.0,215.668810,14.0,391.0,95.442966,67073.0,0.034406
4,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,5,440.0,864.0,501.0,440.0,17.916094,6.061040,...,0.0,117.0,27.463751,15307.0,195.359091,15.0,435.0,121.036614,85958.0,0.047228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2534,422.0,847.0,508.0,422.0,12.093045,6.917525,...,0.0,6.0,1.238990,584.0,13.106635,2.0,26.0,4.784503,5531.0,0.648075
2534,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2535,311.0,770.0,382.0,311.0,11.734377,6.295570,...,0.0,9.0,1.871819,783.0,29.922830,4.0,73.0,12.393099,9306.0,0.431886
2535,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2536,390.0,720.0,460.0,390.0,12.105488,7.714788,...,0.0,7.0,1.221872,503.0,13.748718,2.0,34.0,6.024602,5362.0,0.329752
2536,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2537,323.0,900.0,535.0,323.0,11.135132,7.702116,...,0.0,3.0,0.718026,153.0,5.052632,0.0,14.0,2.488428,1632.0,0.172409


,lif_container_id,lif_image_name,flood_filled,label,area,area_bbox,area_convex,area_filled,axis_major_length,axis_minor_length,...,edCitrine_CTRL_mean_int,edCitrine_CTRL_min_int,edCitrine_CTRL_max_int,edCitrine_CTRL_std_int,edCitrine_CTRL_sum_int,depth,FRET_ratio_sum,FRET_ratio_mean,FRET_ratio_sum_norm,FRET_ratio_mean_norm
0,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,1,344.0,648.0,405.0,344.0,18.542413,5.664366,...,220.125000,31.0,458.0,120.852154,75723.0,0.060050,1.535309,1.535309,0.130684,0.130684
1,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2,466.0,1024.0,524.0,466.0,16.681678,6.095668,...,142.899142,21.0,328.0,80.896244,66591.0,0.047228,1.672433,1.672433,0.174269,0.174269
2,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,3,440.0,800.0,496.0,440.0,16.776578,5.059086,...,170.643182,29.0,366.0,90.749918,75083.0,0.047228,1.556236,1.556236,0.137336,0.137336
3,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,4,311.0,686.0,362.0,311.0,14.512868,6.100066,...,215.668810,14.0,391.0,95.442966,67073.0,0.034406,1.455622,1.455622,0.105355,0.105355
4,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,5,440.0,864.0,501.0,440.0,17.916094,6.061040,...,195.359091,15.0,435.0,121.036614,85958.0,0.047228,1.498336,1.498336,0.118932,0.118932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2534,422.0,847.0,508.0,422.0,12.093045,6.917525,...,13.106635,2.0,26.0,4.784503,5531.0,0.648075,3.106383,3.106383,0.630055,0.630055
2534,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2535,311.0,770.0,382.0,311.0,11.734377,6.295570,...,29.922830,4.0,73.0,12.393099,9306.0,0.431886,2.018041,2.018041,0.284122,0.284122
2535,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2536,390.0,720.0,460.0,390.0,12.105488,7.714788,...,13.748718,2.0,34.0,6.024602,5362.0,0.329752,2.245536,2.245536,0.356432,0.356432
2536,20260129_ABACUS timepoints_sorb 3h,Col0 1,False,2537,323.0,900.0,535.0,323.0,11.135132,7.702116,...,5.052632,0.0,14.0,2.488428,1632.0,0.172409,1.843373,1.843373,0.228603,0.228603
